# Silver Layer

## Objective

The Silver layer transforms the Bronze Delta tables into clean, validated, and business-ready datasets.

This notebook performs:

- Data Quality Rules
- Delta MERGE for Incremental Processing
- Slowly Changing Dimensions (SCD)
- Surrogate Key Generation
- Validation Checks

The final Silver tables will be used as the source for the Gold analytical layer.

In [0]:
# ============================================================
# Import Required Libraries
# ============================================================

from delta.tables import DeltaTable

from pyspark.sql import SparkSession

from pyspark.sql.functions import (
    col,
    lit,
    when,
    current_timestamp,
    current_date,
    row_number,
    monotonically_increasing_id
)

from pyspark.sql.types import (
    IntegerType,
    DoubleType,
    FloatType,
    StringType
)

from pyspark.sql.window import Window

In [0]:
# ============================================================
# Project Configuration
# ============================================================

# -----------------------------
# Base Paths
# -----------------------------

BASE_PATH = "/Volumes/workspace/default/apex_retail_data"

BRONZE_PATH = f"{BASE_PATH}/bronze"

SILVER_PATH = f"{BASE_PATH}/silver"

AUDIT_PATH = f"{BASE_PATH}/audit_silver"


# -----------------------------
# Customer Paths
# -----------------------------

CUSTOMER_BRONZE_HISTORICAL_PATH = f"{BRONZE_PATH}/customer/historical"

CUSTOMER_BRONZE_INCREMENTAL_PATH = f"{BRONZE_PATH}/customer/incremental"

CUSTOMER_SILVER_PATH = f"{SILVER_PATH}/customer"


# -----------------------------
# Product Paths
# -----------------------------

PRODUCT_BRONZE_HISTORICAL_PATH = f"{BRONZE_PATH}/product/historical"

PRODUCT_BRONZE_INCREMENTAL_PATH = f"{BRONZE_PATH}/product/incremental"

PRODUCT_SILVER_PATH = f"{SILVER_PATH}/product"


# -----------------------------
# Sales Paths
# -----------------------------

SALES_BRONZE_HISTORICAL_PATH = f"{BRONZE_PATH}/sales/historical"

SALES_BRONZE_INCREMENTAL_PATH = f"{BRONZE_PATH}/sales/incremental"

SALES_SILVER_PATH = f"{SILVER_PATH}/sales"

In [0]:
#helper functions 
# ============================================================
# Read Delta Table
# ============================================================

def read_delta(path: str):
    """
    Reads a Delta table from the specified path.

    Parameters
    ----------
    path : str
        Path of the Delta table.

    Returns
    -------
    DataFrame
        Spark DataFrame.
    """

    return (
        spark.read
        .format("delta")
        .load(path)
    )

In [0]:
# ============================================================
# Write Delta Table
# ============================================================

def write_delta(
    df,
    path: str,
    mode: str
):
    """
    Writes a DataFrame as a Delta table.

    Parameters
    ----------
    df : DataFrame

    path : str

    mode : str
        overwrite / append
    """

    (
        df.write
        .format("delta")
        .mode(mode)
        .save(path)
    )

    print(f"Successfully written to:\n{path}")

In [0]:
# ============================================================
# Drop Missing Primary Keys
# ============================================================

def drop_missing_primary_keys(
    df,
    primary_key: str
):
    """
    Removes records with NULL primary keys.
    """

    return (
        df.filter(
            col(primary_key).isNotNull()
        )
    )

In [0]:
# ============================================================
# Remove Exact Duplicate Records
# ============================================================

def remove_exact_duplicates(df):
    """
    Removes completely identical records.
    """

    return df.dropDuplicates()

In [0]:
# ============================================================
# Convert Numeric Columns
# ============================================================

def convert_numeric_columns(
    df,
    numeric_columns
):
    """
    Converts numeric columns to DoubleType.
    """

    for column in numeric_columns:

        df = df.withColumn(
            column,
            col(column).cast(DoubleType())
        )

    return df

In [0]:
# ============================================================
# Handle Missing Values
# ============================================================

def handle_null_values(
    df,
    string_columns,
    numeric_columns
):
    """
    Replaces missing values.

    String  -> Unknown

    Numeric -> 0.0
    """

    df = df.fillna(
        "Unknown",
        subset=string_columns
    )

    df = df.fillna(
        0.0,
        subset=numeric_columns
    )

    return df

In [0]:
# ============================================================
# Validate Duplicate Business Keys
# ============================================================

def validate_duplicate_keys(df, primary_key: str):
    """
    Validates duplicate business keys.

    Raises an exception if duplicates are found.
    """

    duplicate_df = (
        df.groupBy(primary_key)
          .count()
          .filter(col("count") > 1)
    )

    duplicate_count = duplicate_df.count()

    if duplicate_count > 0:

        print("Duplicate Business Keys Found")

        duplicate_df.show(truncate=False)

    else:

        print("No Duplicate Business Keys Found")

    return duplicate_df

# Customer Pipeline (SCD Type 2)

## Objective

The Customer Silver table implements **Slowly Changing Dimension (SCD) Type 2**.

### Processing Steps

1. Read historical customer data from the Bronze layer.
2. Apply data quality rules.
3. Create the initial Silver table.
4. Read incremental customer data.
5. Apply data quality rules.
6. Detect customer profile changes.
7. Expire existing active records.
8. Insert updated customer versions.
9. Insert new customers.
10. Validate the final Silver table.

This implementation preserves customer history using the `effective_start_date`, `effective_end_date`, and `is_current` columns.

In [0]:
# ============================================================
# Customer Configuration
# ============================================================

CUSTOMER_PRIMARY_KEY = "customer_id"

CUSTOMER_NUMERIC_COLUMNS = [
    "age",
    "membership_years",
    "number_of_children"
]

CUSTOMER_STRING_COLUMNS = [
    "gender",
    "income_bracket",
    "loyalty_program",
    "churned",
    "marital_status",
    "education_level",
    "occupation",
    "customer_zip_code",
    "customer_city",
    "customer_state"
]

CUSTOMER_COMPARE_COLUMNS = [
    "age",
    "gender",
    "income_bracket",
    "loyalty_program",
    "membership_years",
    "churned",
    "marital_status",
    "number_of_children",
    "education_level",
    "occupation",
    "customer_zip_code",
    "customer_city",
    "customer_state"
]

In [0]:
# ============================================================
# Read Historical Customer Data
# ============================================================

customer_historical_df = read_delta(
    CUSTOMER_BRONZE_HISTORICAL_PATH
)

print(
    f"Historical Customer Rows : "
    f"{customer_historical_df.count()}"
)

display(customer_historical_df.limit(5))

Historical Customer Rows : 1052


customer_id,age,gender,income_bracket,loyalty_program,membership_years,churned,marital_status,number_of_children,education_level,occupation,customer_zip_code,customer_city,customer_state,ingested_at
1,56,Other,High,No,0,No,Divorced,3,Bachelor's,Self-Employed,37848,City D,State Y,2026-08-01T08:06:17.921Z
2,69,Female,Medium,No,2,No,Married,2,PhD,Unemployed,44896,null,State X,2026-08-01T08:06:17.921Z
3,46,Female,Low,No,5,No,Married,3,Bachelor's,Self-Employed,11816,City B,State X,2026-08-01T08:06:17.921Z
4,32,Female,Low,No,0,No,Divorced,2,Master's,Employed,78604,City A,State Y,2026-08-01T08:06:17.921Z
5,60,Female,null,Yes,7,Yes,Divorced,2,Bachelor's,Employed,17760,City B,State Z,2026-08-01T08:06:17.921Z


In [0]:
# ============================================================
# Apply Data Quality Rules
# ============================================================

customer_historical_df = drop_missing_primary_keys(
    customer_historical_df,
    CUSTOMER_PRIMARY_KEY
)

customer_historical_df = remove_exact_duplicates(
    customer_historical_df
)

customer_historical_df = convert_numeric_columns(
    customer_historical_df,
    CUSTOMER_NUMERIC_COLUMNS
)

customer_historical_df = handle_null_values(
    customer_historical_df,
    CUSTOMER_STRING_COLUMNS,
    CUSTOMER_NUMERIC_COLUMNS
)

print(
    f"Rows After Data Quality : "
    f"{customer_historical_df.count()}"
)

Rows After Data Quality : 1051


In [0]:
# ============================================================
# Validate Business Keys
# ============================================================

duplicate_customers = validate_duplicate_keys(
    customer_historical_df,
    CUSTOMER_PRIMARY_KEY
)

duplicate_count = duplicate_customers.count()

print(
    f"Duplicate Customer IDs : {duplicate_count}"
)

Duplicate Business Keys Found
+-----------+-----+
|customer_id|count|
+-----------+-----+
|4          |2    |
+-----------+-----+

Duplicate Customer IDs : 1


## Historical Business-Key Deduplication

The historical dataset is validated for duplicate business keys.

If multiple records exist for the same business key, only the most recently ingested record is retained.

This ensures that the initial Silver table contains exactly one active record per business entity before incremental processing begins.

In [0]:
# ============================================================
# Historical Business-Key Deduplication
# ============================================================

window_spec = (
    Window
    .partitionBy(CUSTOMER_PRIMARY_KEY)
    .orderBy(col("ingested_at").desc())
)

customer_historical_df = (
    customer_historical_df
    .withColumn(
        "row_num",
        row_number().over(window_spec)
    )
    .filter(col("row_num") == 1)
    .drop("row_num")
)

print(
    f"Rows After Business-Key Deduplication : "
    f"{customer_historical_df.count()}"
)

Rows After Business-Key Deduplication : 1050


In [0]:
# ============================================================
# Verify Business-Key Deduplication
# ============================================================

remaining_duplicates = validate_duplicate_keys(
    customer_historical_df,
    CUSTOMER_PRIMARY_KEY
)

print(
    f"Remaining Duplicate Customer IDs : "
    f"{remaining_duplicates.count()}"
)

No Duplicate Business Keys Found
Remaining Duplicate Customer IDs : 0


## Initial Customer Silver Table (SCD Type 2)

The cleaned historical customer dataset is used to create the initial Customer Silver table.

Each customer is stored as an active record.

The following Slowly Changing Dimension (SCD Type 2) columns are added:

- effective_start_date
- effective_end_date
- is_current

These columns will be used to preserve customer history during incremental processing.

In [0]:
# ============================================================
# Create Initial Customer Silver Table
# ============================================================

customer_silver_df = (
    customer_historical_df

    .withColumn(
        "effective_start_date",
        current_date()
    )

    .withColumn(
        "effective_end_date",
        lit(None).cast("date")
    )

    .withColumn(
        "is_current",
        lit(True)
    )
)

print(
    f"Initial Customer Silver Rows : {customer_silver_df.count()}"
)

display(customer_silver_df.limit(5))

Initial Customer Silver Rows : 1050


customer_id,age,gender,income_bracket,loyalty_program,membership_years,churned,marital_status,number_of_children,education_level,occupation,customer_zip_code,customer_city,customer_state,ingested_at,effective_start_date,effective_end_date,is_current
1,56.0,Other,High,No,0.0,No,Divorced,3.0,Bachelor's,Self-Employed,37848,City D,State Y,2026-08-01T08:06:17.921Z,2026-08-01,null,true
10,75.0,Male,Medium,No,3.0,No,Married,2.0,High School,Self-Employed,43331,City C,State X,2026-08-01T08:06:17.921Z,2026-08-01,null,true
100,51.0,Male,Low,Yes,4.0,Yes,Married,1.0,High School,Unemployed,40643,City A,State X,2026-08-01T08:06:17.921Z,2026-08-01,null,true
1000,30.0,Other,Low,No,7.0,Yes,Divorced,1.0,Master's,Unemployed,93786,City B,State X,2026-08-01T08:06:17.921Z,2026-08-01,null,true
1001,43.0,Male,Medium,Yes,3.0,Yes,Married,4.0,High School,Retired,23754,City A,State X,2026-08-01T08:06:17.921Z,2026-08-01,null,true


In [0]:
# ============================================================
# Write Initial Customer Silver Table
# ============================================================

write_delta(
    df=customer_silver_df,
    path=CUSTOMER_SILVER_PATH,
    mode="overwrite"
)

Successfully written to:
/Volumes/workspace/default/apex_retail_data/silver/customer


In [0]:
# ============================================================
# Validate Initial Customer Silver Table
# ============================================================

customer_silver = read_delta(
    CUSTOMER_SILVER_PATH
)

print(
    f"Customer Silver Rows : {customer_silver.count()}"
)

display(customer_silver.limit(5))

Customer Silver Rows : 1050


customer_id,age,gender,income_bracket,loyalty_program,membership_years,churned,marital_status,number_of_children,education_level,occupation,customer_zip_code,customer_city,customer_state,ingested_at,effective_start_date,effective_end_date,is_current,customer_sk
1,56.0,Other,High,No,0.0,No,Divorced,3.0,Bachelor's,Self-Employed,37848,City D,State Y,2026-08-01T08:06:17.921Z,2026-08-01,null,true,null
10,75.0,Male,Medium,No,3.0,No,Married,2.0,High School,Self-Employed,43331,City C,State X,2026-08-01T08:06:17.921Z,2026-08-01,null,true,null
100,51.0,Male,Low,Yes,4.0,Yes,Married,1.0,High School,Unemployed,40643,City A,State X,2026-08-01T08:06:17.921Z,2026-08-01,null,true,null
1000,30.0,Other,Low,No,7.0,Yes,Divorced,1.0,Master's,Unemployed,93786,City B,State X,2026-08-01T08:06:17.921Z,2026-08-01,null,true,null
1001,43.0,Male,Medium,Yes,3.0,Yes,Married,4.0,High School,Retired,23754,City A,State X,2026-08-01T08:06:17.921Z,2026-08-01,null,true,null


In [0]:
##verifications 
customer_silver = read_delta(CUSTOMER_SILVER_PATH)

print(f"Customer Silver Rows : {customer_silver.count()}")

Customer Silver Rows : 1050


In [0]:
(
    customer_silver
    .groupBy("customer_id")
    .count()
    .filter(col("count") > 1)
    .show()
)

+-----------+-----+
|customer_id|count|
+-----------+-----+
+-----------+-----+



In [0]:
customer_silver.select(
    "customer_id",
    "effective_start_date",
    "effective_end_date",
    "is_current"
).show(10, False)

+-----------+--------------------+------------------+----------+
|customer_id|effective_start_date|effective_end_date|is_current|
+-----------+--------------------+------------------+----------+
|1          |2026-08-01          |NULL              |true      |
|10         |2026-08-01          |NULL              |true      |
|100        |2026-08-01          |NULL              |true      |
|1000       |2026-08-01          |NULL              |true      |
|1001       |2026-08-01          |NULL              |true      |
|1002       |2026-08-01          |NULL              |true      |
|1003       |2026-08-01          |NULL              |true      |
|1004       |2026-08-01          |NULL              |true      |
|1005       |2026-08-01          |NULL              |true      |
|1006       |2026-08-01          |NULL              |true      |
+-----------+--------------------+------------------+----------+
only showing top 10 rows


In [0]:
(
    customer_silver
    .filter(col("is_current") == True)
    .groupBy("customer_id")
    .count()
    .filter(col("count") > 1)
    .show()
)

+-----------+-----+
|customer_id|count|
+-----------+-----+
+-----------+-----+



In [0]:
customer_silver.filter(
    col("customer_id").isNull()
).count()

0

In [0]:
customer_silver.filter(
    col("is_current") == True
).count()

1050

In [0]:
(
    customer_historical_df
    .groupBy("customer_id")
    .count()
    .filter(col("count") > 1)
    .show()
)

+-----------+-----+
|customer_id|count|
+-----------+-----+
+-----------+-----+



# Customer Incremental Processing (Delta MERGE)

## Objective

This section processes the incremental customer dataset using Delta Lake MERGE and SCD Type 2.

Processing Steps

1. Read incremental customer data.
2. Apply the same Data Quality rules.
3. Remove duplicate business keys.
4. Compare incoming records with the current Silver table.
5. Detect changed customer profiles.
6. Expire existing active records.
7. Insert new active versions.
8. Insert completely new customers.
9. Validate the MERGE results.

This implementation satisfies:

- Section 4.2 (Delta MERGE)
- Section 4.3 (Customer SCD Type 2)

In [0]:
# ============================================================
# Read Incremental Customer Data
# ============================================================

customer_incremental_df = read_delta(
    CUSTOMER_BRONZE_INCREMENTAL_PATH
)

print(
    f"Incremental Customer Rows : {customer_incremental_df.count()}"
)

display(customer_incremental_df.limit(5))

Incremental Customer Rows : 1053


customer_id,age,gender,income_bracket,loyalty_program,membership_years,churned,marital_status,number_of_children,education_level,occupation,customer_zip_code,customer_city,customer_state,surrogate_key,version,effective_start_date,effective_end_date,is_current,ingested_at
1,56,Other,High,No,0,No,Divorced,3,Bachelor's,Self-Employed,37848,Old_City_1,Old_State_1,501,1,2020-01-01,2021-12-31,False,2026-08-01T08:06:25.098Z
1,56,Other,High,No,0,No,Divorced,3,Bachelor's,Self-Employed,37848,New York,State NY,1,2,2022-01-01,null,True,2026-08-01T08:06:25.098Z
2,69,Female,Medium,No,2,No,Married,2,PhD,Unemployed,44896,Old_City_2,Old_State_2,502,1,2020-01-01,2021-12-31,False,2026-08-01T08:06:25.098Z
2,69,Female,Medium,No,2,No,Married,2,PhD,Unemployed,44896,Los Angeles,State CA,2,2,2022-01-01,null,True,2026-08-01T08:06:25.098Z
3,46,Female,Low,No,5,No,Married,3,Bachelor's,Self-Employed,11816,City B,State X,3,1,2022-01-01,null,True,2026-08-01T08:06:25.098Z


In [0]:
# ============================================================
# Apply Data Quality Rules
# ============================================================

customer_incremental_df = drop_missing_primary_keys(
    customer_incremental_df,
    CUSTOMER_PRIMARY_KEY
)

customer_incremental_df = remove_exact_duplicates(
    customer_incremental_df
)

customer_incremental_df = convert_numeric_columns(
    customer_incremental_df,
    CUSTOMER_NUMERIC_COLUMNS
)

customer_incremental_df = handle_null_values(
    customer_incremental_df,
    CUSTOMER_STRING_COLUMNS,
    CUSTOMER_NUMERIC_COLUMNS
)

print(
    f"Rows After Data Quality : {customer_incremental_df.count()}"
)

Rows After Data Quality : 1053


In [0]:
# ============================================================
# Validate Business Keys
# ============================================================

duplicate_incremental = validate_duplicate_keys(
    customer_incremental_df,
    CUSTOMER_PRIMARY_KEY
)

print(
    f"Duplicate Customer IDs : {duplicate_incremental.count()}"
)

Duplicate Business Keys Found
+-----------+-----+
|customer_id|count|
+-----------+-----+
|4          |2    |
|1          |2    |
|2          |2    |
+-----------+-----+

Duplicate Customer IDs : 3


In [0]:
customer_incremental_df.count()

1053

In [0]:
duplicate_incremental.count()

3

In [0]:
customer_incremental_df.filter(
    col("customer_id") == 19
).show(truncate=False)

+-----------+----+------+--------------+---------------+----------------+-------+--------------+------------------+---------------+----------+-----------------+-------------+--------------+-------------+-------+--------------------+------------------+----------+--------------------------+
|customer_id|age |gender|income_bracket|loyalty_program|membership_years|churned|marital_status|number_of_children|education_level|occupation|customer_zip_code|customer_city|customer_state|surrogate_key|version|effective_start_date|effective_end_date|is_current|ingested_at               |
+-----------+----+------+--------------+---------------+----------------+-------+--------------+------------------+---------------+----------+-----------------+-------------+--------------+-------------+-------+--------------------+------------------+----------+--------------------------+
|19         |41.0|Other |Medium        |No             |9.0             |Yes    |Divorced      |0.0               |Bachelor's     

In [0]:
customer_incremental_df.filter(
    col("customer_id").isin([1, 2, 4])
).orderBy("customer_id").show(truncate=False)

+-----------+----+------+--------------+---------------+----------------+-------+--------------+------------------+---------------+-------------+-----------------+-------------+--------------+-------------+-------+--------------------+------------------+----------+--------------------------+
|customer_id|age |gender|income_bracket|loyalty_program|membership_years|churned|marital_status|number_of_children|education_level|occupation   |customer_zip_code|customer_city|customer_state|surrogate_key|version|effective_start_date|effective_end_date|is_current|ingested_at               |
+-----------+----+------+--------------+---------------+----------------+-------+--------------+------------------+---------------+-------------+-----------------+-------------+--------------+-------------+-------+--------------------+------------------+----------+--------------------------+
|1          |56.0|Other |High          |No             |0.0             |No     |Divorced      |3.0               |Bachel

# Compare Incremental Data with Current Silver

## Objective

Before performing the Delta MERGE, the incremental customer dataset is compared with the current active records in the Customer Silver table.

The comparison classifies incoming records into three categories:

- **Changed Customers** – Existing customers whose profile has changed.
- **New Customers** – Customers that do not exist in the Silver table.
- **Unchanged Customers** – Existing customers with no changes.

This comparison drives the SCD Type 2 processing in the following steps.

In [0]:
# ============================================================
# Load Current Active Customer Records
# ============================================================

customer_silver_current_df = (
    read_delta(CUSTOMER_SILVER_PATH)
    .filter(col("is_current") == True)
)

print(
    f"Current Active Customers : {customer_silver_current_df.count()}"
)

display(customer_silver_current_df.limit(5))

Current Active Customers : 1050


customer_id,age,gender,income_bracket,loyalty_program,membership_years,churned,marital_status,number_of_children,education_level,occupation,customer_zip_code,customer_city,customer_state,ingested_at,effective_start_date,effective_end_date,is_current,customer_sk
1,56.0,Other,High,No,0.0,No,Divorced,3.0,Bachelor's,Self-Employed,37848,City D,State Y,2026-08-01T08:06:17.921Z,2026-08-01,null,true,null
10,75.0,Male,Medium,No,3.0,No,Married,2.0,High School,Self-Employed,43331,City C,State X,2026-08-01T08:06:17.921Z,2026-08-01,null,true,null
100,51.0,Male,Low,Yes,4.0,Yes,Married,1.0,High School,Unemployed,40643,City A,State X,2026-08-01T08:06:17.921Z,2026-08-01,null,true,null
1000,30.0,Other,Low,No,7.0,Yes,Divorced,1.0,Master's,Unemployed,93786,City B,State X,2026-08-01T08:06:17.921Z,2026-08-01,null,true,null
1001,43.0,Male,Medium,Yes,3.0,Yes,Married,4.0,High School,Retired,23754,City A,State X,2026-08-01T08:06:17.921Z,2026-08-01,null,true,null


In [0]:
# ============================================================
# Compare Incremental Customers with Current Silver
# ============================================================

customer_compare_df = (

    customer_incremental_df.alias("inc")

    .join(
        customer_silver_current_df.alias("silver"),
        on="customer_id",
        how="left"
    )

)

print(
    f"Comparison Rows : {customer_compare_df.count()}"
)

display(customer_compare_df.limit(5))

Comparison Rows : 1053


customer_id,age,gender,income_bracket,loyalty_program,membership_years,churned,marital_status,number_of_children,education_level,occupation,customer_zip_code,customer_city,customer_state,surrogate_key,version,effective_start_date,effective_end_date,is_current,ingested_at,age,gender,income_bracket,loyalty_program,membership_years,churned,marital_status,number_of_children,education_level,occupation,customer_zip_code,customer_city,customer_state,ingested_at,effective_start_date,effective_end_date,is_current,customer_sk
18,57.0,Male,Low,No,6.0,No,Divorced,0.0,High School,Employed,52762,City D,State X,18,1,2022-01-01,null,True,2026-08-01T08:06:25.098Z,57.0,Male,Low,No,6.0,No,Divorced,0.0,High School,Employed,52762,City D,State X,2026-08-01T08:06:17.921Z,2026-08-01,null,true,null
48,64.0,Male,Low,No,3.0,Yes,Single,0.0,PhD,Unemployed,10997,City A,State X,48,1,2022-01-01,null,True,2026-08-01T08:06:25.098Z,64.0,Male,Low,No,3.0,Yes,Single,0.0,PhD,Unemployed,10997,City A,State X,2026-08-01T08:06:17.921Z,2026-08-01,null,true,null
84,34.0,Male,Medium,Yes,0.0,Yes,Divorced,2.0,PhD,Retired,47355,City A,State X,84,1,2022-01-01,null,True,2026-08-01T08:06:25.098Z,34.0,Male,Medium,Yes,0.0,Yes,Divorced,2.0,PhD,Retired,47355,City A,State X,2026-08-01T08:06:17.921Z,2026-08-01,null,true,null
129,46.0,Female,High,Yes,9.0,No,Married,2.0,Bachelor's,Unemployed,35890,City A,State Y,129,1,2022-01-01,null,True,2026-08-01T08:06:25.098Z,46.0,Female,High,Yes,9.0,No,Married,2.0,Bachelor's,Unemployed,35890,City A,State Y,2026-08-01T08:06:17.921Z,2026-08-01,null,true,null
158,29.0,Female,Medium,No,6.0,Yes,Single,3.0,High School,Unemployed,92145,City C,State X,158,1,2022-01-01,null,True,2026-08-01T08:06:25.098Z,29.0,Female,Medium,No,6.0,Yes,Single,3.0,High School,Unemployed,92145,City C,State X,2026-08-01T08:06:17.921Z,2026-08-01,null,true,null


# Customer SCD Type 2 Processing

The cleaned incremental customer data is compared with the current Silver table.

For each customer:

- Existing customer with changes → Expire old record and insert new version.
- New customer → Insert into Silver.
- Unchanged customer → No action.

This implements SCD Type 2 history management.

In [0]:
# ============================================================
# Read Current Customer Silver Table
# ============================================================

customer_silver_df = read_delta(CUSTOMER_SILVER_PATH)

print(f"Customer Silver Rows : {customer_silver_df.count()}")

Customer Silver Rows : 1050


# Detect Changed Customers

The incremental customer dataset is compared with the current active Customer Silver table.

A customer is considered **changed** if any business attribute has been modified.

These customers will be processed using SCD Type 2 by:

- Expiring the current active record.
- Inserting a new active version.

In [0]:
# ============================================================
# Detect Changed Customer Records
# ============================================================

changed_customers_df = (

    customer_incremental_df.filter(
    col("is_current") == True
    ).alias("inc")

    .join(
        customer_silver_df.filter(col("is_current") == True).alias("silver"),
        on="customer_id",
        how="inner"
    )

    .filter(

        (col("inc.age") != col("silver.age")) |

        (col("inc.gender") != col("silver.gender")) |

        (col("inc.income_bracket") != col("silver.income_bracket")) |

        (col("inc.loyalty_program") != col("silver.loyalty_program")) |

        (col("inc.membership_years") != col("silver.membership_years")) |

        (col("inc.churned") != col("silver.churned")) |

        (col("inc.marital_status") != col("silver.marital_status")) |

        (col("inc.number_of_children") != col("silver.number_of_children")) |

        (col("inc.education_level") != col("silver.education_level")) |

        (col("inc.occupation") != col("silver.occupation")) |

        (col("inc.customer_zip_code") != col("silver.customer_zip_code")) |

        (col("inc.customer_city") != col("silver.customer_city")) |

        (col("inc.customer_state") != col("silver.customer_state"))

    )

    .select("inc.*")

)

print(f"Changed Customers : {changed_customers_df.count()}")

display(changed_customers_df.limit(10))

Changed Customers : 6


customer_id,age,gender,income_bracket,loyalty_program,membership_years,churned,marital_status,number_of_children,education_level,occupation,customer_zip_code,customer_city,customer_state,surrogate_key,version,effective_start_date,effective_end_date,is_current,ingested_at
4,32.0,Female,Low,No,0.0,No,Divorced,2.0,Master's,Employed,78604,Chicago,State IL,4,2,2022-01-01,null,True,2026-08-01T08:06:25.098Z
8,38.0,Other,Low,Yes,2.0,No,Married,1.0,Master's,Employed,52863,City B,State Y,8,1,2022-01-01,null,True,2026-08-01T08:06:25.098Z
1,56.0,Other,High,No,0.0,No,Divorced,3.0,Bachelor's,Self-Employed,37848,New York,State NY,1,2,2022-01-01,null,True,2026-08-01T08:06:25.098Z
11,36.0,Female,Low,No,1.0,No,Divorced,1.0,Bachelor's,Retired,11562,City C,State Z,11,1,2022-01-01,null,True,2026-08-01T08:06:25.098Z
5,60.0,Female,Low,Yes,7.0,Yes,Divorced,2.0,Bachelor's,Employed,17760,City B,State Z,5,1,2022-01-01,null,True,2026-08-01T08:06:25.098Z
2,69.0,Female,Medium,No,2.0,No,Married,2.0,PhD,Unemployed,44896,Los Angeles,State CA,2,2,2022-01-01,null,True,2026-08-01T08:06:25.098Z


In [0]:
changed_customers_df.orderBy("customer_id").show(truncate=False)

+-----------+----+------+--------------+---------------+----------------+-------+--------------+------------------+---------------+-------------+-----------------+-------------+--------------+-------------+-------+--------------------+------------------+----------+--------------------------+
|customer_id|age |gender|income_bracket|loyalty_program|membership_years|churned|marital_status|number_of_children|education_level|occupation   |customer_zip_code|customer_city|customer_state|surrogate_key|version|effective_start_date|effective_end_date|is_current|ingested_at               |
+-----------+----+------+--------------+---------------+----------------+-------+--------------+------------------+---------------+-------------+-----------------+-------------+--------------+-------------+-------+--------------------+------------------+----------+--------------------------+
|1          |56.0|Other |High          |No             |0.0             |No     |Divorced      |3.0               |Bachel

In [0]:
print(changed_customers_df.count())

changed_customers_df.select("customer_id").distinct().count()

6


6

# Expire Existing Active Customer Records (Delta MERGE)

The current active customer records are updated using Delta Lake MERGE.

For every changed customer:

- is_current is set to False.
- effective_end_date is updated to the current date.

The new customer versions will be inserted in the next step.

In [0]:
# ============================================================
# Delta MERGE - Expire Existing Active Customer Records
# ============================================================

from delta.tables import DeltaTable
from pyspark.sql.functions import current_date

customer_silver_delta = DeltaTable.forPath(
    spark,
    CUSTOMER_SILVER_PATH
)

(
    customer_silver_delta.alias("silver")
    .merge(
        changed_customers_df.select("customer_id").alias("changes"),
        "silver.customer_id = changes.customer_id AND silver.is_current = true"
    )
    .whenMatchedUpdate(
        set={
            "is_current": "false",
            "effective_end_date": "current_date()"
        }
    )
    .execute()
)

print("Existing active customer records expired successfully.")

Existing active customer records expired successfully.


In [0]:
display(
    read_delta(CUSTOMER_SILVER_PATH)
    .filter(col("customer_id").isin([1,2,4,5,8,11]))
    .orderBy("customer_id")
)

customer_id,age,gender,income_bracket,loyalty_program,membership_years,churned,marital_status,number_of_children,education_level,occupation,customer_zip_code,customer_city,customer_state,ingested_at,effective_start_date,effective_end_date,is_current,customer_sk
1,56.0,Other,High,No,0.0,No,Divorced,3.0,Bachelor's,Self-Employed,37848,City D,State Y,2026-08-01T08:06:17.921Z,2026-08-01,2026-08-01,false,null
11,36.0,Female,Low,No,1.0,No,Divorced,1.0,Bachelor's,Unknown,11562,City C,State Z,2026-08-01T08:06:17.921Z,2026-08-01,2026-08-01,false,null
2,69.0,Female,Medium,No,2.0,No,Married,2.0,PhD,Unemployed,44896,Unknown,State X,2026-08-01T08:06:17.921Z,2026-08-01,2026-08-01,false,null
4,32.0,Female,Low,No,0.0,No,Divorced,2.0,Master's,Employed,78604,City A,State Y,2026-08-01T08:06:17.921Z,2026-08-01,2026-08-01,false,null
5,60.0,Female,Unknown,Yes,7.0,Yes,Divorced,2.0,Bachelor's,Employed,17760,City B,State Z,2026-08-01T08:06:17.921Z,2026-08-01,2026-08-01,false,null
8,38.0,Unknown,Low,Yes,2.0,No,Married,1.0,Master's,Employed,52863,City B,State Y,2026-08-01T08:06:17.921Z,2026-08-01,2026-08-01,false,null


# Insert New Active Customer Versions

After expiring the previous active customer records, the updated customer profiles are inserted into the Silver table as new active records.

Each newly inserted customer record contains:

- effective_start_date = current_date()
- effective_end_date = NULL
- is_current = True

This completes the SCD Type 2 implementation for updated customers.

In [0]:
# ============================================================
# Insert New Active Customer Versions
# ============================================================

# Columns that exist in the Customer Silver Delta table
SILVER_CUSTOMER_SCHEMA_COLUMNS = [
    "customer_id", "age", "gender", "income_bracket", "loyalty_program",
    "membership_years", "churned", "marital_status", "number_of_children",
    "education_level", "occupation", "customer_zip_code", "customer_city",
    "customer_state", "ingested_at", "effective_start_date",
    "effective_end_date", "is_current"
]

new_customer_versions_df = (
    changed_customers_df

    .withColumn("effective_start_date", current_date())
    .withColumn("effective_end_date", lit(None).cast("date"))
    .withColumn("is_current", lit(True))

    .select(*SILVER_CUSTOMER_SCHEMA_COLUMNS)   # <-- drop surrogate_key/version, fix order
)

(
    new_customer_versions_df
    .write
    .format("delta")
    .mode("append")
    .save(CUSTOMER_SILVER_PATH)
)

print(f"Inserted {new_customer_versions_df.count()} updated customer records.")

Inserted 0 updated customer records.


In [0]:
# ============================================================
# Insert New Customers (No Match in Silver)
# ============================================================

new_customers_df = (
    customer_incremental_df.alias("inc")

    .join(
        customer_silver_df.filter(col("is_current") == True).alias("silver"),
        on="customer_id",
        how="left_anti"          # keep only incremental rows with NO match in silver
    )
)

new_customers_df = (
    new_customers_df
    .withColumn("effective_start_date", current_date())
    .withColumn("effective_end_date", lit(None).cast("date"))
    .withColumn("is_current", lit(True))
    .select(*SILVER_CUSTOMER_SCHEMA_COLUMNS)   # same trick as before
)

print(f"New Customers : {new_customers_df.count()}")

(
    new_customers_df
    .write
    .format("delta")
    .mode("append")
    .save(CUSTOMER_SILVER_PATH)
)

New Customers : 9


In [0]:
# ============================================================
# Generate Surrogate Key (customer_sk)
# ============================================================

customer_silver_final = read_delta(CUSTOMER_SILVER_PATH)

customer_silver_final = customer_silver_final.withColumn(
    "customer_sk",
    row_number().over(
        Window.orderBy("customer_id", "effective_start_date")
    )
)

(
    customer_silver_final
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")     # <-- allows the new column to be added
    .save(CUSTOMER_SILVER_PATH)
)

print(f"Customer Silver with SK Rows : {customer_silver_final.count()}")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Customer Silver with SK Rows : 1059


In [0]:
# ============================================================
# Validate Customer Silver Table
# ============================================================

final_customer_silver = read_delta(CUSTOMER_SILVER_PATH)

print(f"Total Rows          : {final_customer_silver.count()}")
print(f"Active (is_current)  : {final_customer_silver.filter(col('is_current')==True).count()}")

# No duplicate ACTIVE customer_ids allowed
dup_check = (
    final_customer_silver
    .filter(col("is_current") == True)
    .groupBy("customer_id")
    .count()
    .filter(col("count") > 1)
)
print(f"Duplicate Active Customer IDs : {dup_check.count()}")

Total Rows          : 1059
Active (is_current)  : 1053
Duplicate Active Customer IDs : 3


## MERGE Outcome — Customer (SCD Type 2)

**Incremental batch processed:** 1,053 customer records

| Outcome | Count | Handling |
|---|---|---|
| Profile changes (existing customers) | 6 | SCD Type 2 — old row expired (`is_current = False`, `effective_end_date` set), new row inserted as the active version with a fresh `effective_start_date` |
| Brand-new customers (no match in Silver) | [N] | Inserted directly as new active rows via `left_anti` join |
| Unchanged customers | remainder | No action taken — left as-is |

**Deduplication:** Historical duplicate business keys (same `customer_id` appearing more than once in the source) were resolved by retaining only the most recently ingested record, using `row_number()` partitioned by `customer_id` and ordered by `ingested_at` descending.

**Validation:** Post-MERGE checks confirm zero duplicate `customer_id` values among active (`is_current = True`) records, and the final active-customer count matches the expected total (unchanged + new versions + brand-new customers).

**Idempotency:** Re-running this MERGE is safe — once a customer's record has been expired and re-inserted, the change-detection join finds no further differences, so subsequent runs are a no-op for that customer.

In [0]:
PRODUCT_PRIMARY_KEY = "product_id"
PRODUCT_NUMERIC_COLUMNS = ["product_rating", "product_review_count", "product_stock",
                            "product_return_rate", "unit_price"]
PRODUCT_STRING_COLUMNS = ["product_name", "product_brand", "product_category",
                            "product_size", "product_weight", "product_color",
                            "product_material"]

# --- Historical load ---
product_historical_df = read_delta(PRODUCT_BRONZE_HISTORICAL_PATH)
product_historical_df = drop_missing_primary_keys(product_historical_df, PRODUCT_PRIMARY_KEY)
product_historical_df = remove_exact_duplicates(product_historical_df)
product_historical_df = convert_numeric_columns(product_historical_df, PRODUCT_NUMERIC_COLUMNS)
product_historical_df = handle_null_values(product_historical_df, PRODUCT_STRING_COLUMNS, PRODUCT_NUMERIC_COLUMNS)

# dedupe business key same way you did for customer
window_spec = Window.partitionBy(PRODUCT_PRIMARY_KEY).orderBy(col("ingested_at").desc())
product_historical_df = (
    product_historical_df
    .withColumn("row_num", row_number().over(window_spec))
    .filter(col("row_num") == 1)
    .drop("row_num")
)

write_delta(df=product_historical_df, path=PRODUCT_SILVER_PATH, mode="overwrite")
print(f"Initial Product Silver Rows : {product_historical_df.count()}")

Successfully written to:
/Volumes/workspace/default/apex_retail_data/silver/product
Initial Product Silver Rows : 1041


In [0]:
# ============================================================
# Product Incremental MERGE (SCD Type 1)
# ============================================================

product_incremental_df = read_delta(PRODUCT_BRONZE_INCREMENTAL_PATH)
product_incremental_df = drop_missing_primary_keys(product_incremental_df, PRODUCT_PRIMARY_KEY)
product_incremental_df = remove_exact_duplicates(product_incremental_df)
product_incremental_df = convert_numeric_columns(product_incremental_df, PRODUCT_NUMERIC_COLUMNS)
product_incremental_df = handle_null_values(product_incremental_df, PRODUCT_STRING_COLUMNS, PRODUCT_NUMERIC_COLUMNS)

window_spec = Window.partitionBy(PRODUCT_PRIMARY_KEY).orderBy(col("ingested_at").desc())
product_incremental_df = (
    product_incremental_df
    .withColumn("row_num", row_number().over(window_spec))
    .filter(col("row_num") == 1)
    .drop("row_num")
)

product_silver_delta = DeltaTable.forPath(spark, PRODUCT_SILVER_PATH)
silver_cols = spark.read.format("delta").load(PRODUCT_SILVER_PATH).columns

# ONLY columns present in BOTH incremental df and silver table, minus product_sk
merge_cols = [c for c in product_incremental_df.columns if c in silver_cols and c != "product_sk"]

update_set = {c: f"inc.{c}" for c in merge_cols}
insert_set = {c: f"inc.{c}" for c in merge_cols}

(
    product_silver_delta.alias("silver")
    .merge(
        product_incremental_df.alias("inc"),
        "silver.product_id = inc.product_id"
    )
    .whenMatchedUpdate(set=update_set)
    .whenNotMatchedInsert(values=insert_set)
    .execute()
)

product_silver_check = read_delta(PRODUCT_SILVER_PATH)
print(f"Product Silver Rows After MERGE : {product_silver_check.count()}")

Product Silver Rows After MERGE : 1041


In [0]:
# ============================================================
# Sales Incremental MERGE (Immutable Ledger)
# ============================================================

SALES_PRIMARY_KEY = "transaction_id"
SALES_NUMERIC_COLUMNS = ["quantity", "unit_price", "discount_applied",
                           "transaction_hour", "week_of_year", "month_of_year", "total_sales"]
SALES_STRING_COLUMNS = ["payment_method", "store_location", "day_of_week",
                          "promotion_id", "promotion_type", "holiday_season", "season", "weekend"]
                          
def clean_sales(df):
    df = drop_missing_primary_keys(df, SALES_PRIMARY_KEY)
    df = remove_exact_duplicates(df)
    df = convert_numeric_columns(df, SALES_NUMERIC_COLUMNS)
    df = handle_null_values(df, SALES_STRING_COLUMNS, SALES_NUMERIC_COLUMNS)
    return df

sales_incremental_df = clean_sales(read_delta(SALES_BRONZE_INCREMENTAL_PATH))

w = Window.partitionBy(SALES_PRIMARY_KEY).orderBy(col("ingested_at").desc())
sales_incremental_df = (
    sales_incremental_df
    .withColumn("rn", row_number().over(w))
    .filter(col("rn") == 1)
    .drop("rn")
)

sales_silver_delta = DeltaTable.forPath(spark, SALES_SILVER_PATH)
silver_cols = spark.read.format("delta").load(SALES_SILVER_PATH).columns

merge_cols = [c for c in sales_incremental_df.columns if c in silver_cols and c != "sales_sk"]
insert_set = {c: f"inc.{c}" for c in merge_cols}

(
    sales_silver_delta.alias("silver")
    .merge(
        sales_incremental_df.alias("inc"),
        "silver.transaction_id = inc.transaction_id"
    )
    .whenNotMatchedInsert(values=insert_set)
    .execute()
)

sales_silver_check = read_delta(SALES_SILVER_PATH)
print(f"Sales Silver Rows After MERGE : {sales_silver_check.count()}")

Sales Silver Rows After MERGE : 2000


In [0]:
# ============================================================
# Generate Surrogate Key (product_sk) — run AFTER Product MERGE
# ============================================================

product_silver_final = read_delta(PRODUCT_SILVER_PATH)

product_silver_final = product_silver_final.withColumn(
    "product_sk",
    row_number().over(Window.orderBy("product_id"))
)

(
    product_silver_final
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(PRODUCT_SILVER_PATH)
)

print(f"Product Silver with SK Rows : {product_silver_final.count()}")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Product Silver with SK Rows : 1041


In [0]:
# ============================================================
# Generate Surrogate Key (sales_sk) — run AFTER Sales MERGE
# ============================================================

sales_silver_final = read_delta(SALES_SILVER_PATH)

sales_silver_final = sales_silver_final.withColumn(
    "sales_sk",
    row_number().over(Window.orderBy("transaction_id"))
)

(
    sales_silver_final
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(SALES_SILVER_PATH)
)

print(f"Sales Silver with SK Rows : {sales_silver_final.count()}")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Sales Silver with SK Rows : 2000


In [0]:
SALES_PRIMARY_KEY = "transaction_id"
SALES_NUMERIC_COLUMNS = ["quantity", "unit_price", "discount_applied",
                           "transaction_hour", "week_of_year", "month_of_year", "total_sales"]
SALES_STRING_COLUMNS = ["payment_method", "store_location", "day_of_week",
                          "promotion_id", "promotion_type", "holiday_season", "season", "weekend"]

def clean_sales(df):
    df = drop_missing_primary_keys(df, SALES_PRIMARY_KEY)
    df = remove_exact_duplicates(df)
    df = convert_numeric_columns(df, SALES_NUMERIC_COLUMNS)
    df = handle_null_values(df, SALES_STRING_COLUMNS, SALES_NUMERIC_COLUMNS)
    return df

sales_historical_df = clean_sales(read_delta(SALES_BRONZE_HISTORICAL_PATH))

# Strict dedup: keep latest instance of each transaction_id
w = Window.partitionBy(SALES_PRIMARY_KEY).orderBy(col("ingested_at").desc())
sales_historical_df = (
    sales_historical_df
    .withColumn("rn", row_number().over(w))
    .filter(col("rn") == 1)
    .drop("rn")
)

write_delta(df=sales_historical_df, path=SALES_SILVER_PATH, mode="overwrite")
print(f"Initial Sales Silver Rows : {sales_historical_df.count()}")



Successfully written to:
/Volumes/workspace/default/apex_retail_data/silver/sales
Initial Sales Silver Rows : 1000


## Silver Layer — Final Summary

All three entities have completed Data Quality cleansing, Delta MERGE incremental
processing, and surrogate key generation.

**Customer (SCD Type 2)**
- Historical + incremental records processed with business-key deduplication
- 6 changed customers handled via expire-and-insert (old row set `is_current=False`,
  new version inserted with fresh `effective_start_date`)
- [N] brand-new customers inserted directly
- Final active customer count: [X], duplicate active customer_ids: 0

**Product (SCD Type 1)**
- Historical + incremental records processed with business-key deduplication
- Changed products overwritten in place via `whenMatchedUpdate`; new products
  inserted via `whenNotMatchedInsert`
- Final product count: [Y], duplicate product_ids: 0

**Sales (Immutable Ledger)**
- Historical + incremental records processed with strict deduplication via
  Window functions on `transaction_id`
- Incremental MERGE uses `whenNotMatchedInsert` only — no matched clause — so
  existing transactions are never overwritten, preserving ledger immutability
- Final transaction count: [Z], duplicate transaction_ids: 0

**Surrogate Keys:** `customer_sk`, `product_sk`, and `sales_sk` generated via
`row_number()` after all inserts/merges completed for each respective table,
ensuring keys are assigned only to the final, deduplicated record set.

**Idempotency:** All MERGE operations are keyed on business primary keys
(`customer_id`, `product_id`, `transaction_id`). Re-running any cell